In [9]:
# CELL 1: Check GPU
import subprocess
print(subprocess.getoutput('nvidia-smi'))

Thu Jun 11 10:14:37 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8              9W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [10]:
INPUT_DIR     = '/kaggle/input'
WORKING_DIR   = '/kaggle/working'

# Corrected paths based on actual file locations
AVEC_ZIP      = None  # not needed — dataset is already extracted
BASELINE_CKPT = '/kaggle/input/datasets/kashmalaamer/avec2014-checkpoints/c3d_avec2014_best.pth'
GENDER_CSV    = '/kaggle/input/datasets/kashmalaamer/avec2014-gender/avec2014_gender.csv'

# Dataset is already extracted — point directly to it
EXTRACT_PATH  = '/kaggle/input/datasets/kashmalaamer/avec2014/AVEC2014'
DATA_ROOT     = '/kaggle/input/datasets/kashmalaamer/avec2014/AVEC2014/AVEC2014'
FRAME_ROOT    = os.path.join(WORKING_DIR, 'avec2014_frames')
QR_CKPT       = os.path.join(WORKING_DIR, 'c3d_avec2014_quantile.pth')

CLIP_LEN    = 16
STRIDE      = 8
BATCH_SIZE  = 4
N_QUANTILES = 99
M_BINS      = 4
ALPHA       = 0.1

print(f'BASELINE_CKPT exists: {os.path.exists(BASELINE_CKPT)}')
print(f'GENDER_CSV    exists: {os.path.exists(GENDER_CSV)}')
print(f'DATA_ROOT     exists: {os.path.exists(DATA_ROOT)}')

BASELINE_CKPT exists: True
GENDER_CSV    exists: True
DATA_ROOT     exists: True


In [11]:
# CELL 3: Keep-alive
import time, threading

def keep_alive():
    while True:
        time.sleep(60)
        print('.', end='', flush=True)

t = threading.Thread(target=keep_alive, daemon=True)
t.start()
print('Keep-alive started.')

Keep-alive started.


In [12]:
# CELL 4: No unzipping needed — Kaggle dataset is already extracted
print('Dataset already available at:', DATA_ROOT)
print('labels.csv exists:', os.path.exists(os.path.join(DATA_ROOT, 'labels.csv')))

Dataset already available at: /kaggle/input/datasets/kashmalaamer/avec2014/AVEC2014/AVEC2014
labels.csv exists: True


In [13]:
# CELL 5: Load labels and gender map
def load_labels():
    df = pd.read_csv(os.path.join(DATA_ROOT, 'labels.csv'))
    labels = {}
    for _, row in df.iterrows():
        key = str(row['filename']).strip().replace('\\', '/')
        key = os.path.splitext(key)[0]
        labels[key] = float(row['BDI-II'])
    print(f'Labels loaded: {len(labels)}')
    return labels

def load_gender_map():
    # Try multiple locations
    for path in [
        os.path.join(DATA_ROOT, 'gender.csv'),
        GENDER_CSV,
    ]:
        if os.path.exists(path):
            df = pd.read_csv(path)
            gmap = {}
            for _, row in df.iterrows():
                key = str(row['filename']).strip().replace('\\', '/')
                key = os.path.splitext(key)[0]
                gmap[key] = str(row['gender']).strip().upper()
            print(f'Gender map loaded: {len(gmap)} entries from {path}')
            return gmap
    print('No gender.csv found.')
    return {}

labels     = load_labels()
gender_map = load_gender_map()

Labels loaded: 300
Gender map loaded: 300 entries from /kaggle/input/datasets/kashmalaamer/avec2014-gender/avec2014_gender.csv


In [14]:
# CELL 6: Collect videos
def collect_videos(folders, labels):
    if isinstance(folders, str):
        folders = [folders]
    items = []
    for folder in folders:
        if not os.path.exists(folder):
            print(f'WARNING: not found: {folder}')
            continue
        for root, _, files in os.walk(folder):
            for f in sorted(files):
                if not f.lower().endswith('.mp4'):
                    continue
                path = os.path.join(root, f)
                rel  = os.path.relpath(path, DATA_ROOT).replace('\\', '/')
                stem = os.path.splitext(rel)[0]
                if stem in labels:
                    items.append((path, stem, labels[stem]))
    return items

TRAIN_DIRS = [
    os.path.join(DATA_ROOT, 'Training'),
    os.path.join(DATA_ROOT, 'Development'),
]
CAL_DIR  = os.path.join(DATA_ROOT, 'Testing', 'Northwind')
TEST_DIR = os.path.join(DATA_ROOT, 'Testing', 'Freeform')

train_items = collect_videos(TRAIN_DIRS, labels)
cal_items   = collect_videos(CAL_DIR,    labels)
test_items  = collect_videos(TEST_DIR,   labels)

print(f'Train : {len(train_items)}  (expected 200)')
print(f'Cal   : {len(cal_items)}    (expected 50)')
print(f'Test  : {len(test_items)}   (expected 50)')

train_labels_arr = np.array([l for _, _, l in train_items])
LABEL_MEAN = float(train_labels_arr.mean())
LABEL_STD  = float(train_labels_arr.std())
print(f'Label norm — mean: {LABEL_MEAN:.2f}  std: {LABEL_STD:.2f}')

Train : 200  (expected 200)
Cal   : 50    (expected 50)
Test  : 50   (expected 50)
Label norm — mean: 15.34  std: 12.07


In [15]:
# CELL 7: Pre-extract frames (skips if already done)
os.makedirs(FRAME_ROOT, exist_ok=True)

def extract_video(path, stem):
    out_dir = os.path.join(FRAME_ROOT, stem.replace('/', '_'))
    if os.path.exists(out_dir) and len(os.listdir(out_dir)) >= CLIP_LEN:
        return
    os.makedirs(out_dir, exist_ok=True)
    cap = cv2.VideoCapture(path)
    idx = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        frame = cv2.resize(frame, (112, 112))
        cv2.imwrite(
            os.path.join(out_dir, f'{idx:05d}.jpg'),
            frame,
            [cv2.IMWRITE_JPEG_QUALITY, 95]
        )
        idx += 1
    cap.release()

all_items = train_items + cal_items + test_items
print(f'Extracting frames for {len(all_items)} videos...')
for path, stem, label in tqdm(all_items):
    extract_video(path, stem)
print('Extraction complete.')

Extracting frames for 300 videos...


 17%|█▋        | 51/300 [00:59<03:58,  1.04it/s]

.

 29%|██▉       | 88/300 [01:57<06:55,  1.96s/it]

.

...

.

 99%|█████████▊| 296/300 [06:58<00:07,  1.86s/it]

.

100%|██████████| 300/300 [07:11<00:00,  1.44s/it]

Extraction complete.


In [16]:
# CELL 8: Dataset classes

def load_frames_from_ssd(stem, start, n=CLIP_LEN):
    out_dir = os.path.join(FRAME_ROOT, stem.replace('/', '_'))
    frames  = []
    for i in range(start, start + n):
        fpath = os.path.join(out_dir, f'{i:05d}.jpg')
        if not os.path.exists(fpath):
            break
        frame = cv2.imread(fpath)
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frames.append(frame)
    return frames

def count_frames(stem):
    out_dir = os.path.join(FRAME_ROOT, stem.replace('/', '_'))
    if not os.path.exists(out_dir):
        return 0
    return len([f for f in os.listdir(out_dir) if f.endswith('.jpg')])

def frames_to_tensor(frames):
    arr = np.stack(frames, axis=0).astype(np.float32) / 255.0
    arr = (arr - MEAN) / STD
    arr = arr.transpose(3, 0, 1, 2)
    return torch.from_numpy(arr)

def get_clip_starts(T, stride=STRIDE):
    return list(range(0, T - CLIP_LEN + 1, stride))


class AVEC2014Train(Dataset):
    def __init__(self, items):
        self.samples = []
        for path, stem, label in items:
            T = count_frames(stem)
            if T < CLIP_LEN:
                continue
            for s in get_clip_starts(T):
                self.samples.append((stem, s, label))
        print(f'Training clips: {len(self.samples)}')

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        stem, start, label = self.samples[idx]
        frames = load_frames_from_ssd(stem, start, CLIP_LEN)
        if len(frames) < CLIP_LEN:
            while len(frames) < CLIP_LEN:
                frames.append(frames[-1])
        norm_label = (label - LABEL_MEAN) / LABEL_STD
        return frames_to_tensor(frames), torch.tensor(norm_label, dtype=torch.float32)


class AVEC2014Eval(Dataset):
    def __init__(self, items):
        self.samples = []
        for path, stem, label in items:
            T = count_frames(stem)
            if T >= CLIP_LEN:
                self.samples.append((stem, label))
        print(f'Eval videos: {len(self.samples)}')

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        stem, label = self.samples[idx]
        T     = count_frames(stem)
        clips = []
        for s in get_clip_starts(T, stride=8):
            frames = load_frames_from_ssd(stem, s, CLIP_LEN)
            if len(frames) < CLIP_LEN:
                continue
            clips.append(frames_to_tensor(frames))
        if not clips:
            clips.append(torch.zeros(3, CLIP_LEN, 112, 112))
        return torch.stack(clips), torch.tensor(label, dtype=torch.float32), stem

In [17]:
# CELL 9: C3D Quantile Regression Model
# Paper: final FC = 99 units, quantiles 0.01 to 0.99

class C3DQuantile(nn.Module):
    def __init__(self, n_quantiles=99, dropout=0.5):
        super().__init__()
        self.conv1  = nn.Conv3d(3,   64,  kernel_size=(3,3,3), padding=(1,1,1))
        self.pool1  = nn.MaxPool3d(kernel_size=(1,2,2), stride=(1,2,2))
        self.conv2  = nn.Conv3d(64,  128, kernel_size=(3,3,3), padding=(1,1,1))
        self.pool2  = nn.MaxPool3d(kernel_size=(2,2,2), stride=(2,2,2))
        self.conv3a = nn.Conv3d(128, 256, kernel_size=(3,3,3), padding=(1,1,1))
        self.conv3b = nn.Conv3d(256, 256, kernel_size=(3,3,3), padding=(1,1,1))
        self.pool3  = nn.MaxPool3d(kernel_size=(2,2,2), stride=(2,2,2))
        self.conv4a = nn.Conv3d(256, 512, kernel_size=(3,3,3), padding=(1,1,1))
        self.conv4b = nn.Conv3d(512, 512, kernel_size=(3,3,3), padding=(1,1,1))
        self.pool4  = nn.MaxPool3d(kernel_size=(2,2,2), stride=(2,2,2))
        self.conv5a = nn.Conv3d(512, 512, kernel_size=(3,3,3), padding=(1,1,1))
        self.conv5b = nn.Conv3d(512, 512, kernel_size=(3,3,3), padding=(1,1,1))
        self.pool5  = nn.MaxPool3d(kernel_size=(2,2,2), stride=(2,2,2), padding=(0,1,1))
        self.relu    = nn.ReLU(inplace=True)
        self.fc6     = nn.Linear(8192, 4096)
        self.fc7     = nn.Linear(4096, 64)
        self.fc8     = nn.Linear(64, n_quantiles)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        x = self.relu(self.conv1(x));  x = self.pool1(x)
        x = self.relu(self.conv2(x));  x = self.pool2(x)
        x = self.relu(self.conv3a(x))
        x = self.relu(self.conv3b(x)); x = self.pool3(x)
        x = self.relu(self.conv4a(x))
        x = self.relu(self.conv4b(x)); x = self.pool4(x)
        x = self.relu(self.conv5a(x))
        x = self.relu(self.conv5b(x)); x = self.pool5(x)
        x = x.view(x.size(0), -1)
        x = self.dropout(self.relu(self.fc6(x)))
        x = self.dropout(self.relu(self.fc7(x)))
        return self.fc8(x)

qr_model = C3DQuantile(n_quantiles=N_QUANTILES).to(device)
print(f'Parameters: {sum(p.numel() for p in qr_model.parameters()):,}')
with torch.no_grad():
    out = qr_model(torch.zeros(2, 3, 16, 112, 112).to(device))
    print(f'Forward pass OK — output: {out.shape}')

Parameters: 61,483,107
Forward pass OK — output: torch.Size([2, 99])


In [18]:
# CELL 10: Load baseline checkpoint into quantile model

FEATURE_MAP = {
    'features.0.':   'conv1.',
    'features.3.':   'conv2.',
    'features.6.':   'conv3a.',
    'features.8.':   'conv3b.',
    'features.11.':  'conv4a.',
    'features.13.':  'conv4b.',
    'features.16.':  'conv5a.',
    'features.18.':  'conv5b.',
    'classifier.0.': 'fc6.',
    'classifier.3.': 'fc7.',
}

def load_baseline_into_quantile(qr_model, path):
    if not os.path.exists(path):
        print(f'WARNING: not found: {path}')
        return
    ckpt = torch.load(path, map_location=device)
    if isinstance(ckpt, dict) and 'state_dict' in ckpt:
        ckpt = ckpt['state_dict']
    model_dict = qr_model.state_dict()
    matched = {}
    for k, v in ckpt.items():
        new_k = k
        for old_prefix, new_prefix in FEATURE_MAP.items():
            if k.startswith(old_prefix):
                new_k = k.replace(old_prefix, new_prefix)
                break
        if 'fc8' in new_k:
            continue
        if new_k in model_dict and model_dict[new_k].shape == v.shape:
            matched[new_k] = v
    model_dict.update(matched)
    qr_model.load_state_dict(model_dict)
    print(f'Baseline layers loaded: {len(matched)} tensors matched')

load_baseline_into_quantile(qr_model, BASELINE_CKPT)

Baseline layers loaded: 20 tensors matched


In [19]:
# CELL 11: Build datasets and DataLoader
train_dataset = AVEC2014Train(train_items)
cal_dataset   = AVEC2014Eval(cal_items)
test_dataset  = AVEC2014Eval(test_items)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=False,
)
print(f'Batches per epoch: {len(train_loader)}')

Training clips: 37961
Eval videos: 50
Eval videos: 50
Batches per epoch: 9491


In [20]:
# CELL 12: Pinball loss and evaluation functions

QUANTILES = torch.linspace(0.01, 0.99, N_QUANTILES).to(device)

def pinball_loss(preds, targets):
    targets = targets.unsqueeze(1)
    q       = QUANTILES.unsqueeze(0)
    errors  = targets - preds
    loss    = torch.where(errors >= 0, q * errors, (q - 1) * errors)
    return loss.mean()

def get_video_quantiles(dataset, model, chunk_size=8):
    model.eval()
    results = []
    with torch.no_grad():
        for clips, label, stem in dataset:
            all_preds = []
            for i in range(0, len(clips), chunk_size):
                chunk = clips[i:i+chunk_size].to(device)
                preds = model(chunk)
                all_preds.append(preds.cpu())
                torch.cuda.empty_cache()
            q_norm = torch.cat(all_preds).mean(dim=0).numpy()
            q_pred = q_norm * LABEL_STD + LABEL_MEAN
            results.append((stem, label.item(), q_pred))
    return results

def eval_point_prediction(results):
    y_true = np.array([r[1] for r in results])
    y_pred = np.array([r[2][49] for r in results])
    mae    = np.mean(np.abs(y_true - y_pred))
    rmse   = np.sqrt(np.mean((y_true - y_pred)**2))
    return mae, rmse

print('Functions defined.')

Functions defined.


In [21]:
# CELL 13: Train quantile regression model

for p in qr_model.parameters():
    p.requires_grad = True

optimizer    = torch.optim.Adam(qr_model.parameters(), lr=1e-5, weight_decay=1e-4)
best_cal_mae = float('inf')
patience     = 10
no_improve   = 0

print('=== Quantile Regression Training ===')
for epoch in range(1, 41):
    qr_model.train()
    total_loss, n_batches = 0.0, 0
    for clips, targets in train_loader:
        clips   = clips.to(device)
        targets = targets.to(device)
        loss    = pinball_loss(qr_model(clips), targets)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        n_batches  += 1
    torch.cuda.empty_cache()
    avg_loss = total_loss / n_batches
    cal_results      = get_video_quantiles(cal_dataset, qr_model)
    cal_mae, cal_rmse = eval_point_prediction(cal_results)
    print(f'Epoch {epoch:3d} | Pinball: {avg_loss:.4f} | Cal MAE: {cal_mae:.4f} | Cal RMSE: {cal_rmse:.4f}')
    if cal_mae < best_cal_mae:
        best_cal_mae = cal_mae
        no_improve   = 0
        torch.save(qr_model.state_dict(), QR_CKPT)
        print(f'  -> Best saved (Cal MAE={best_cal_mae:.4f})')
    else:
        no_improve += 1
        if no_improve >= patience:
            print(f'  Early stopping at epoch {epoch}.')
            break
print(f'Best Cal MAE: {best_cal_mae:.4f}')

=== Quantile Regression Training ===
.............................................................Epoch   1 | Pinball: 0.1997 | Cal MAE: 7.6257 | Cal RMSE: 9.6560
  -> Best saved (Cal MAE=7.6257)
.............................................................Epoch   2 | Pinball: 0.0970 | Cal MAE: 7.8263 | Cal RMSE: 9.7387
.............................................................Epoch   3 | Pinball: 0.0716 | Cal MAE: 7.6975 | Cal RMSE: 9.4771
.............................................................Epoch   4 | Pinball: 0.0619 | Cal MAE: 8.2899 | Cal RMSE: 10.4255
.............................................................Epoch   5 | Pinball: 0.0550 | Cal MAE: 8.0110 | Cal RMSE: 9.9282
..............................................................Epoch   6 | Pinball: 0.0512 | Cal MAE: 7.7386 | Cal RMSE: 9.9061
.............................................................Epoch   7 | Pinball: 0.0494 | Cal MAE: 7.5786 | Cal RMSE: 9.7113
  -> Best saved (Cal MAE=7.5786)
.............

KeyboardInterrupt: 

In [22]:
import os
ckpt = '/kaggle/working/c3d_avec2014_quantile.pth'
print(f'Exists: {os.path.exists(ckpt)}')
print(f'Size: {os.path.getsize(ckpt)/1e6:.1f} MB')

Exists: True
Size: 245.9 MB


In [23]:
# This makes it downloadable from the Files panel on the right
from IPython.display import FileLink
FileLink('/kaggle/working/c3d_avec2014_quantile.pth')

/kaggle/working/c3d_avec2014_quantile.pth

...

In [24]:
import shutil, os

# Copy checkpoint to a guaranteed output location
shutil.copy(
    '/kaggle/working/c3d_avec2014_quantile.pth',
    '/kaggle/working/quantile_checkpoint_epoch7.pth'
)
print('Copied.')

# Also copy any results if they exist
for f in os.listdir('/kaggle/working'):
    print(f, os.path.getsize(os.path.join('/kaggle/working', f))/1e6, 'MB')

Copied.
.virtual_documents 0.004096 MB
avec2014_frames 0.02048 MB
quantile_checkpoint_epoch7.pth 245.940649 MB
c3d_avec2014_quantile.pth 245.940649 MB
........

In [ ]:
# RECOVERY CELL — only run if session disconnected during Cell 13
# Re-run Cells 1-12 first, then run this

# qr_model.load_state_dict(torch.load(QR_CKPT, map_location=device))
# for p in qr_model.parameters():
#     p.requires_grad = True
# optimizer = torch.optim.Adam(qr_model.parameters(), lr=1e-5, weight_decay=1e-4)
# print('Resuming...')
# # Then re-run the training loop in Cell 13

In [25]:
# CELL 14: Load best model and get predictions
qr_model.load_state_dict(torch.load(QR_CKPT, map_location=device))
print(f'Loaded: {QR_CKPT}')

cal_results  = get_video_quantiles(cal_dataset,  qr_model)
test_results = get_video_quantiles(test_dataset, qr_model)

cal_mae,  cal_rmse  = eval_point_prediction(cal_results)
test_mae, test_rmse = eval_point_prediction(test_results)
print(f'Cal  MAE={cal_mae:.4f}  RMSE={cal_rmse:.4f}')
print(f'Test MAE={test_mae:.4f}  RMSE={test_rmse:.4f}')

Loaded: /kaggle/working/c3d_avec2014_quantile.pth
...............Cal  MAE=7.5786  RMSE=9.7113
Test MAE=8.4299  RMSE=10.4762


In [27]:
# CELL 15: CQR
Q_LO_IDX = int(ALPHA / 2 * N_QUANTILES)
Q_HI_IDX = int((1 - ALPHA / 2) * N_QUANTILES) - 1
print(f'Lower quantile: {QUANTILES[Q_LO_IDX].item():.2f}  Upper: {QUANTILES[Q_HI_IDX].item():.2f}')

cal_scores = np.array([
    max(q[Q_LO_IDX] - y, y - q[Q_HI_IDX])
    for _, y, q in cal_results
])
beta = np.quantile(cal_scores, 1 - ALPHA)
print(f'Global beta = {beta:.4f}')

test_intervals_cqr = [
    (stem, y, q[Q_LO_IDX] - beta, q[Q_HI_IDX] + beta)
    for stem, y, q in test_results
]
picp_cqr = np.mean([lo <= y <= hi for _, y, lo, hi in test_intervals_cqr])
mpiw_cqr = np.mean([hi - lo for _, _, lo, hi in test_intervals_cqr])
print(f'CQR: PICP={picp_cqr:.4f}  MPIW={mpiw_cqr:.4f}')

Lower quantile: 0.05  Upper: 0.94
Global beta = 14.9148
CQR: PICP=0.8800  MPIW=36.5307


In [28]:
# CELL 16: Group Conditional Conformal Prediction
# M=4 equal-mass bins, initialise r_hat_{m,s} = global beta

cal_data = pd.DataFrame([{
    'stem':   stem,
    'y_true': y,
    'y_lo':   q[Q_LO_IDX],
    'y_hi':   q[Q_HI_IDX],
    'r':      max(q[Q_LO_IDX] - y, y - q[Q_HI_IDX]),
    'gender': gender_map.get(stem, '?')
} for stem, y, q in cal_results])

print(cal_data['gender'].value_counts())

cal_sorted = cal_data.sort_values('y_true').reset_index(drop=True)
N_cal, bin_size = len(cal_sorted), len(cal_sorted) // M_BINS
bins = []
for m in range(M_BINS):
    s  = m * bin_size
    e  = (m + 1) * bin_size if m < M_BINS - 1 else N_cal
    bd = cal_sorted.iloc[s:e]
    bins.append({'m': m, 'l': bd['y_true'].min(), 'u': bd['y_true'].max(), 'data': bd})
    print(f'Bin {m+1}: [{bd["y_true"].min():.1f}, {bd["y_true"].max():.1f}]  N={len(bd)}')

GENDERS = ['F', 'M']
r_hat   = {(m, s): beta for m in range(M_BINS) for s in GENDERS}

def avg_coverage(bins, r_hat, gender):
    covs = []
    for m, b in enumerate(bins):
        g = b['data'][b['data']['gender'] == gender]
        if len(g) == 0:
            continue
        r = r_hat[(m, gender)]
        c = sum(1 for _, row in g.iterrows() if (row['y_lo']-r) <= row['y_true'] <= (row['y_hi']+r))
        covs.append(c / len(g))
    return np.mean(covs) if covs else 0.0

for s in GENDERS:
    print(f'Initial coverage {"Female" if s=="F" else "Male"}: {avg_coverage(bins, r_hat, s):.4f}')

gender
M    30
F    20
Name: count, dtype: int64
Bin 1: [0.0, 3.0]  N=12
Bin 2: [3.0, 12.0]  N=12
Bin 3: [12.0, 21.0]  N=12
Bin 4: [22.0, 43.0]  N=14
Initial coverage Female: 0.9000
Initial coverage Male: 0.9097


In [29]:
# CELL 17: Fairness-Aware Optimization

TARGET = 1 - ALPHA

def slope_down(bin_data, r, gender):
    g = bin_data[bin_data['gender'] == gender]
    if len(g) == 0: return 0.0
    scores = sorted(g['r'].tolist())
    below  = [s for s in scores if s < r]
    if not below: return 0.0
    return (1.0/len(g)) / (r - below[-1] + 1e-8)

def slope_up(bin_data, r, gender):
    g = bin_data[bin_data['gender'] == gender]
    if len(g) == 0: return float('inf')
    scores = sorted(g['r'].tolist())
    above  = [s for s in scores if s > r]
    if not above: return float('inf')
    return (1.0/len(g)) / (above[0] - r + 1e-8)

print('=== Fairness-Aware Optimization ===')
for iteration in range(500):
    cov   = {s: avg_coverage(bins, r_hat, s) for s in GENDERS}
    if all(abs(cov[s] - TARGET) < 0.02 for s in GENDERS):
        print(f'Converged at iteration {iteration}.')
        break
    over  = max(GENDERS, key=lambda s: cov[s])
    under = min(GENDERS, key=lambda s: cov[s])
    if cov[over] <= TARGET:
        for s in GENDERS:
            for m in range(M_BINS):
                if slope_up(bins[m]['data'], r_hat[(m,s)], s) < float('inf'):
                    r_hat[(m,s)] += 0.1
        continue
    sd_best = max(range(M_BINS), key=lambda m: slope_down(bins[m]['data'], r_hat[(m,over)],  over))
    su_best = min(range(M_BINS), key=lambda m: slope_up(bins[m]['data'],   r_hat[(m,under)], under))
    gd = slope_down(bins[sd_best]['data'], r_hat[(sd_best,over)],  over)
    gu = slope_up(bins[su_best]['data'],   r_hat[(su_best,under)], under)
    if gu > gd and cov[under] >= TARGET - 0.02:
        print(f'Slope condition met at iteration {iteration}.')
        break
    r_hat[(sd_best,over)]  -= 0.1
    r_hat[(su_best,under)] += 0.1
    if iteration % 50 == 0:
        print(f'Iter {iteration:4d} | F: {cov["F"]:.4f}  M: {cov["M"]:.4f}')

final_cov = {s: avg_coverage(bins, r_hat, s) for s in GENDERS}
for s in GENDERS:
    print(f'  {"Female" if s=="F" else "Male":6s}: {final_cov[s]:.4f}  (target {TARGET:.2f})')
print(f'PICP Gap: {abs(final_cov["F"]-final_cov["M"]):.4f}')

=== Fairness-Aware Optimization ===
Converged at iteration 0.
  Female: 0.9000  (target 0.90)
  Male  : 0.9097  (target 0.90)
PICP Gap: 0.0097


In [30]:
# CELL 18: Apply FUQ to test set

def fuq_interval(y_lo, y_hi, gender, bins, r_hat):
    ulo, uhi = float('inf'), float('-inf')
    for m, b in enumerate(bins):
        r  = r_hat[(m, gender)]
        lo = max(y_lo-r, b['l'])
        hi = min(y_hi+r, b['u'])
        if lo <= hi:
            ulo = min(ulo, lo)
            uhi = max(uhi, hi)
    if ulo == float('inf'):
        r   = r_hat[(0, gender)]
        ulo = y_lo - r
        uhi = y_hi + r
    return ulo, uhi

test_fuq = []
for stem, y_true, q_pred in test_results:
    g      = gender_map.get(stem, 'M')
    lo, hi = fuq_interval(q_pred[Q_LO_IDX], q_pred[Q_HI_IDX], g, bins, r_hat)
    test_fuq.append({'stem': stem, 'y_true': y_true, 'y_pred': q_pred[49],
                     'lo': lo, 'hi': hi, 'gender': g, 'covered': lo<=y_true<=hi})

test_fuq_df = pd.DataFrame(test_fuq)
print('===== FUQ Results =====')
print(f'Overall PICP : {test_fuq_df["covered"].mean():.4f}  (target {1-ALPHA:.2f})')
print(f'Overall MPIW : {(test_fuq_df["hi"]-test_fuq_df["lo"]).mean():.4f}')
picps = {}
for s in GENDERS:
    g = test_fuq_df[test_fuq_df['gender']==s]
    if len(g)==0: continue
    ps = g['covered'].mean()
    picps[s] = ps
    print(f'  {"Female" if s=="F" else "Male":6s} N={len(g):3d}  PICP={ps:.4f}  MPIW={(g["hi"]-g["lo"]).mean():.4f}')
print(f'PICP Gap: {abs(picps.get("F",0)-picps.get("M",0)):.4f}')

===== FUQ Results =====
Overall PICP : 0.8800  (target 0.90)
Overall MPIW : 30.1956
  Female N= 15  PICP=0.8667  MPIW=29.1211
  Male   N= 35  PICP=0.8857  MPIW=30.6562
PICP Gap: 0.0190
.

In [31]:
# CELL 19: Comparison table CQR vs FUQ
test_cqr_df = pd.DataFrame([{
    'stem': stem, 'y_true': y, 'lo': lo, 'hi': hi,
    'gender': gender_map.get(stem,'M'), 'covered': lo<=y<=hi
} for stem, y, lo, hi in test_intervals_cqr])

print(f'{"Method":8s} {"PICP":>8s} {"MPIW":>8s} {"PICP(F)":>10s} {"PICP(M)":>10s} {"Gap":>8s}')
print('-'*60)
for name, df in [("CQR", test_cqr_df), ("FUQ", test_fuq_df)]:
    pa = df['covered'].mean()
    mw = (df['hi']-df['lo']).mean()
    pf = df[df['gender']=='F']['covered'].mean() if (df['gender']=='F').any() else float('nan')
    pm = df[df['gender']=='M']['covered'].mean() if (df['gender']=='M').any() else float('nan')
    print(f'{name:8s} {pa:8.4f} {mw:8.4f} {pf:10.4f} {pm:10.4f} {abs(pf-pm):8.4f}')

Method       PICP     MPIW    PICP(F)    PICP(M)      Gap
------------------------------------------------------------
CQR        0.8800  36.5307     0.8667     0.8857   0.0190
FUQ        0.8800  30.1956     0.8667     0.8857   0.0190


In [33]:
# CELL 20: Save results to working directory
# Download from Kaggle output tab after running

test_fuq_df.to_csv(os.path.join(WORKING_DIR, 'avec2014_fuq_results.csv'), index=False)
pd.DataFrame([{'m':m,'gender':s,'r_hat':r_hat[(m,s)]} for m in range(M_BINS) for s in GENDERS]
).to_csv(os.path.join(WORKING_DIR, 'avec2014_fuq_thresholds.csv'), index=False)

print('Results saved to /kaggle/working/')
print('Download from the Kaggle output tab on the right.')
print()
print('===== Final Summary =====')
print(f'QR  Test MAE  : {test_mae:.4f}')
print(f'CQR PICP      : {test_cqr_df["covered"].mean():.4f}')
print(f'CQR MPIW      : {(test_cqr_df["hi"]-test_cqr_df["lo"]).mean():.4f}')
cqr_gap = abs(test_cqr_df[test_cqr_df.gender=='F']['covered'].mean() -
              test_cqr_df[test_cqr_df.gender=='M']['covered'].mean())
fuq_gap = abs(test_fuq_df[test_fuq_df.gender=='F']['covered'].mean() -
              test_fuq_df[test_fuq_df.gender=='M']['covered'].mean())
print(f'CQR PICP Gap  : {cqr_gap:.4f}')
print(f'FUQ PICP      : {test_fuq_df["covered"].mean():.4f}')
print(f'FUQ MPIW      : {(test_fuq_df["hi"]-test_fuq_df["lo"]).mean():.4f}')
print(f'FUQ PICP Gap  : {fuq_gap:.4f}')

Results saved to /kaggle/working/
Download from the Kaggle output tab on the right.

===== Final Summary =====
QR  Test MAE  : 8.4299
CQR PICP      : 0.8800
CQR MPIW      : 36.5307
CQR PICP Gap  : 0.0190
FUQ PICP      : 0.8800
FUQ MPIW      : 30.1956
FUQ PICP Gap  : 0.0190


In [34]:
import pandas as pd
df = pd.read_csv('/kaggle/input/datasets/kashmalaamer/avec2014-gender/avec2014_gender.csv')
print(df.columns.tolist())
print(df.head(10))
print(df['age_group'].value_counts())
print(df['gender'].value_counts())

['filename', 'gender', 'age', 'age_group']
                                 filename gender   age age_group
0  Training/Freeform/203_1_Freeform_video      M  26.1     Young
1  Training/Freeform/205_2_Freeform_video      F  33.5       Old
2  Training/Freeform/207_2_Freeform_video      F  34.4       Old
3  Training/Freeform/208_2_Freeform_video      F  29.7     Young
4  Training/Freeform/209_1_Freeform_video      M  28.9     Young
5  Training/Freeform/213_1_Freeform_video      M  28.5     Young
6  Training/Freeform/214_1_Freeform_video      M  24.6     Young
7  Training/Freeform/215_2_Freeform_video      F  29.5     Young
8  Training/Freeform/215_3_Freeform_video      F  29.3     Young
9  Training/Freeform/217_2_Freeform_video      M  29.3     Young
age_group
Old      200
Young    100
Name: count, dtype: int64
gender
M    195
F    105
Name: count, dtype: int64
...

In [35]:
import os
for f in os.listdir('/kaggle/working'):
    size = os.path.getsize(os.path.join('/kaggle/working', f)) / 1e6
    print(f'{f}  {size:.1f} MB')

.virtual_documents  0.0 MB
avec2014_frames  0.0 MB
avec2014_fuq_thresholds.csv  0.0 MB
avec2014_fuq_results.csv  0.0 MB
quantile_checkpoint_epoch7.pth  245.9 MB
c3d_avec2014_quantile.pth  245.9 MB
.

In [37]:
import pandas as pd
df = pd.read_csv('/kaggle/working/avec2014_fuq_results.csv')
print(f'Rows: {len(df)}')
print(df.head(3))

Rows: 50
                                    stem  y_true     y_pred        lo  \
0  Testing/Freeform/203_2_Freeform_video     8.0  16.127270  0.000000   
1  Testing/Freeform/206_2_Freeform_video     3.0  22.951504  4.238663   
2  Testing/Freeform/210_2_Freeform_video     1.0   6.455982  0.000000   

          hi gender  covered  
0  34.140419      M     True  
1  42.349377      M    False  
2  24.099998      M     True  
.

In [38]:
from IPython.display import FileLink
import os

# List all important files
files = [
    '/kaggle/working/avec2014_fuq_results.csv',
    '/kaggle/working/avec2014_fuq_thresholds.csv', 
    '/kaggle/working/quantile_checkpoint_epoch7.pth'
]

for f in files:
    size = os.path.getsize(f) / 1e6
    print(f'{size:.1f} MB — {f}')

0.0 MB — /kaggle/working/avec2014_fuq_results.csv
0.0 MB — /kaggle/working/avec2014_fuq_thresholds.csv
245.9 MB — /kaggle/working/quantile_checkpoint_epoch7.pth
...

In [39]:
import subprocess
# Create dataset from working directory file
result = subprocess.getoutput('''
kaggle datasets init -p /kaggle/working
''')
print(result)

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/urllib3/connection.py", line 198, in _new_conn
    sock = connection.create_connection(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/urllib3/util/connection.py", line 60, in create_connection
    for res in socket.getaddrinfo(host, port, family, socket.SOCK_STREAM):
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/socket.py", line 978, in getaddrinfo
    for res in _socket.getaddrinfo(host, port, family, type, proto, flags):
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
socket.gaierror: [Errno -3] Temporary failure in name resolution

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/urllib3/connectionpool.py", line 787, in urlopen
    response = self._make_request(
             

In [41]:
# Install required library
!pip install pydrive2 -q

from pydrive2.auth import GoogleAuth
from pydrive2.drive import GoogleDrive
from google.colab import auth
auth.authenticate_user()
from oauth2client.client import GoogleCredentials

gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive = GoogleDrive(gauth)

# Upload checkpoint
f = drive.CreateFile({'title': 'quantile_checkpoint_epoch7.pth'})
f.SetContentFile('/kaggle/working/quantile_checkpoint_epoch7.pth')
f.Upload()
print('Checkpoint uploaded')

# Upload CSV
f2 = drive.CreateFile({'title': 'avec2014_fuq_results.csv'})
f2.SetContentFile('/kaggle/working/avec2014_fuq_results.csv')
f2.Upload()
print('CSV uploaded')

Checkpoint uploaded
CSV uploaded
.

In [42]:
import pandas as pd
import numpy as np

# Load gender CSV for age data
df_gender = pd.read_csv('/kaggle/input/datasets/kashmalaamer/avec2014-gender/avec2014_gender.csv')

# Load FUQ results
test_fuq_df = pd.read_csv('/kaggle/working/avec2014_fuq_results.csv')

# Build age map
age_map = dict(zip(df_gender['filename'], df_gender['age_group']))

# Add age and subgroup columns
test_fuq_df['age_group'] = test_fuq_df['stem'].map(age_map)
test_fuq_df['subgroup']  = test_fuq_df['gender'] + '_' + test_fuq_df['age_group'].fillna('Unknown')

print('=== Test Set Subgroup Distribution ===')
print(test_fuq_df['subgroup'].value_counts())

print('\n=== FUQ Results by Gender ===')
print(f'{"Group":15s} {"N":>5s} {"PICP":>8s} {"MPIW":>8s}')
print('-' * 40)
for g in ['F', 'M']:
    grp = test_fuq_df[test_fuq_df['gender'] == g]
    lbl = 'Female' if g == 'F' else 'Male'
    print(f'{lbl:15s} {len(grp):>5d} {grp["covered"].mean():>8.4f} {(grp["hi"]-grp["lo"]).mean():>8.4f}')

print('\n=== FUQ Results by Age Group ===')
print(f'{"Group":15s} {"N":>5s} {"PICP":>8s} {"MPIW":>8s}')
print('-' * 40)
for ag in ['Young', 'Old']:
    grp = test_fuq_df[test_fuq_df['age_group'] == ag]
    if len(grp) == 0:
        continue
    print(f'{ag:15s} {len(grp):>5d} {grp["covered"].mean():>8.4f} {(grp["hi"]-grp["lo"]).mean():>8.4f}')

print('\n=== FUQ Results by Gender x Age Subgroup ===')
print(f'{"Subgroup":20s} {"N":>5s} {"PICP":>8s} {"MPIW":>8s} {"MAE":>8s}')
print('-' * 55)
for sg in ['F_Young', 'F_Old', 'M_Young', 'M_Old']:
    grp = test_fuq_df[test_fuq_df['subgroup'] == sg]
    if len(grp) == 0:
        continue
    mae = (grp['y_true'] - grp['y_pred']).abs().mean()
    lbl = sg.replace('F_', 'Female_').replace('M_', 'Male_')
    print(f'{lbl:20s} {len(grp):>5d} {grp["covered"].mean():>8.4f} {(grp["hi"]-grp["lo"]).mean():>8.4f} {mae:>8.4f}')

print('\n=== PICP Gaps ===')
picps = {}
for sg in ['F_Young', 'F_Old', 'M_Young', 'M_Old']:
    grp = test_fuq_df[test_fuq_df['subgroup'] == sg]
    if len(grp) > 0:
        picps[sg] = grp['covered'].mean()

for pair in [('F_Young','M_Young'), ('F_Old','M_Old'),
             ('F_Young','F_Old'), ('M_Young','M_Old'),
             ('F_Young','M_Old'), ('F_Old','M_Young')]:
    a, b = pair
    la = a.replace('F_','Female_').replace('M_','Male_')
    lb = b.replace('F_','Female_').replace('M_','Male_')
    gap = abs(picps.get(a,0) - picps.get(b,0))
    print(f'{la:20s} vs {lb:20s} gap: {gap:.4f}')

=== Test Set Subgroup Distribution ===
subgroup
M_Old      23
F_Old      13
M_Young    12
F_Young     2
Name: count, dtype: int64

=== FUQ Results by Gender ===
Group               N     PICP     MPIW
----------------------------------------
Female             15   0.8667  29.1211
Male               35   0.8857  30.6562

=== FUQ Results by Age Group ===
Group               N     PICP     MPIW
----------------------------------------
Young              14   0.8571  27.7912
Old                36   0.8889  31.1307

=== FUQ Results by Gender x Age Subgroup ===
Subgroup                 N     PICP     MPIW      MAE
-------------------------------------------------------
Female_Young             2   1.0000  25.1418   1.3871
Female_Old              13   0.8462  29.7333   9.1199
Male_Young              12   0.8333  28.2328   7.5534
Male_Old                23   0.9130  31.9205   9.1096

=== PICP Gaps ===
Female_Young         vs Male_Young           gap: 0.1667
Female_Old           vs Male_Old   